# Solar filaments: native 2048 × 2048 YOLO + SAM 2

One notebook, no local Python modules. **Run in order.** Start with `MODE="smoke"`,
inspect the merge preview, then change to `"full"` and use a fresh run directory.
All experiment settings are in the next code cell. A CUDA GPU is recommended;
reduce model size, image batch size or prompt batch size if memory is insufficient.
The pipeline never responds to memory pressure by resizing or cropping images.

**Workflow:** union annotators per source → connected filament masks → grouped split
→ YOLO box training → SAM 2 decoder fine-tuning → validation-only threshold search
→ full-resolution COCO RLE submission. Original annotator masks remain the validation
reference. Training union components and original annotated instances have different
semantics; touching filaments can merge. Inspect this before a long run.

Input images stay 2048 × 2048 throughout loading, augmentation and both model inputs.
SAM 2 is explicitly configured for this size and called directly, bypassing its
resizing predictor. Internal feature downsampling and logit upsampling still exist.
Pretrained SAM 2 was trained at a different resolution; native input alone does not
guarantee recovery of thin filaments. Fine-tuning and held-out validation are essential.

**Default hardware profile:** Kaggle **GPU T4 x2**, four CPU workers total.
Training uses model-level concurrency; it does not synchronize one model across
both GPUs. Inference uses one full pipeline per GPU. Select the dual-T4 accelerator
and restart the kernel before running. This hardware profile is not tested here.


In [ ]:
# YOLO_SAM_RUNTIME
import os
from pathlib import Path

# EDIT THIS CELL. No other cell contains experiment hyperparameters.
MODE = "smoke"                          # "smoke" (execution check) or "full"
CFG = {
    "data_root": "",                    # auto-discover locally or under /kaggle/input
    "output_root": "/kaggle/working" if os.name == "posix" and Path("/kaggle").exists() else "./outputs",
    "run_name": "solar_yolo_sam_t4x2",         # change for a new experiment
    "seed": 10,
    "native_size": 2048,                 # invariant: changing this is rejected
    "resources": {
        "gpu_ids": [0, 1],              # Kaggle: GPU T4 x2; select [0] for one GPU
        "cpu_workers": 4,              # TOTAL loader workers / CPU jobs, not per GPU
        "torch_threads": 1,             # avoid 4 workers each starting a CPU thread pool
        "prefetch_factor": 2,
        "pin_memory": True,
        "vram_budget_gib": 13.0,         # allocator cap per T4; leave room for CUDA/context
        "progress_seconds": 15, "job_timeout_hours": 10,
        "stream_logs": True,            # show worker stdout/stderr live in this cell
        "log_poll_seconds": 0.5,         # seconds between incremental log reads
    },
    "amp": True,                        # FP16 on T4; no BF16 requirement
    "split": {"val_fraction": 0.20, "multi_annotator_train_only": True},
    "merge": {"connectivity": 8, "min_component_area": 1},
    "smoke": {"train_sources": 2, "val_sources": 2, "test_sources": 2},
    "stages": {"train_yolo": True, "train_sam": True, "submission": True},
    "yolo": {
        "weights": "yolo26n.pt",          # pretrained detection model, not -seg
        "epochs": 60, "batch": 2, "lr0": 0.001,  # images on the detector GPU
        "optimizer": "AdamW", "weight_decay": 0.0005, "patience": 15,
        "flip_lr": 0.5, "flip_ud": 0.5, "resume": False,
        "channels_last": True, "deterministic": True,
        "cache": False, "cos_lr": True, "pretrained": True,
        "hsv_v": 0.3, "plots": True, "verbose": True,
    },
    "sam": {
        "weights": "sam2.1_t.pt",         # SAM 2 / 2.1 only; use a local .pt offline
        "epochs": 15, "lr": 0.0001, "weight_decay": 0.0001,
        "prompt_batch": 2, "accumulate_images": 4, "patience": 5,
        "box_pad": 8, "box_jitter": 4,    # pixels; padding also used for inference
        "flip_lr": 0.5, "flip_ud": 0.5,
        "bce_weight": 1.0, "dice_weight": 1.0, "iou_weight": 0.1,
        "positive_weight": 3.0, "grad_clip": 1.0, "resume": False,
        "log_every_images": 16,          # SAM training/validation progress within each epoch
    },
    "predict": {"proposal_conf": 0.05, "box_iou": 0.7, "max_det": 100},
    "search": {
        "det_conf": [0.05, 0.15, 0.25], "mask_threshold": [0.4, 0.5, 0.6],
        "mask_nms_iou": [0.8], "min_area": [1],
    },
    "review_images": 3,
}

# Training gallery layout. Missing SDK artifacts are skipped with a clear note.
YOLO_VISUAL_GROUPS = {
    "Overall Performance & Loss Tracking": ["results.png"],
    "Bounding Box Localization Metrics": ["BoxF1_curve.png", "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png"],
    "Instance Segmentation Mask Metrics": ["MaskF1_curve.png", "MaskPR_curve.png", "MaskP_curve.png", "MaskR_curve.png"],
    "Confusion Matrices": ["confusion_matrix.png", "confusion_matrix_normalized.png"],
    "Labels": ["labels.jpg"],
    "Training Dataset Batches & Anchors": ["train_batch0.jpg", "train_batch1.jpg", "train_batch2.jpg"],
    "Validation Predictions vs Ground Truth": ["val_batch0_labels.jpg", "val_batch0_pred.jpg",
        "val_batch1_labels.jpg", "val_batch1_pred.jpg", "val_batch2_labels.jpg", "val_batch2_pred.jpg"],
}
DISPLAY_SETTINGS = {
    "Overall Performance & Loss Tracking": {"cols": 1, "col_width": 16, "row_height": 9},
    "Bounding Box Localization Metrics": {"cols": 2, "col_width": 8, "row_height": 6},
    "Instance Segmentation Mask Metrics": {"cols": 2, "col_width": 8, "row_height": 6},
    "Confusion Matrices": {"cols": 1, "col_width": 10, "row_height": 8},
    "Labels": {"cols": 1, "col_width": 12, "row_height": 8},
    "Training Dataset Batches & Anchors": {"cols": 3, "col_width": 6, "row_height": 5},
    "Validation Predictions vs Ground Truth": {"cols": 2, "col_width": 6, "row_height": 5},
}


## 1. Dependencies and run identity

Use Python 3.12 with compatible CUDA PyTorch and torchvision already installed.
Select Kaggle **GPU T4 x2**; this profile requires Linux.
The cell installs missing auxiliary packages and pins Ultralytics because this
notebook uses its low-level SAM 2 API. It leaves the installed PyTorch build alone.
Internet is needed initially for packages and pretrained weights; attach both `.pt`
files and install dependencies in advance for an offline Kaggle run.

In [ ]:
# YOLO_SAM_RUNTIME
# Bound numerical libraries before importing torch/numpy in this kernel.
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[variable] = str(CFG["resources"]["torch_threads"])

import importlib.util
import importlib.metadata
import subprocess
import sys

DEPENDENCIES = {
    "ultralytics": "ultralytics==8.4.198", "pycocotools": "pycocotools>=2.0.7,<3",
    "scipy": "scipy>=1.11,<2", "matplotlib": "matplotlib>=3.8,<4",
    "pandas": "pandas>=2.1,<4", "cv2": "opencv-python-headless>=4.8,<6",
}
missing = [package for module, package in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
if importlib.metadata.version("ultralytics") != "8.4.198":
    raise RuntimeError("Install ultralytics==8.4.198 and restart the kernel before running.")

import ast
import csv
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, DataLoader, Sampler
import gc
import hashlib
import itertools
import json
import math
import os
import random
import shutil
import time
from collections import defaultdict
from contextlib import nullcontext

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pycocotools import mask as coco_mask
from scipy import ndimage as ndi
from ultralytics import SAM, YOLO, settings

assert MODE in {"smoke", "full"}
assert CFG["native_size"] == 2048, "Native 2048-pixel input is an invariant."
assert CFG["merge"]["connectivity"] in (4, 8)
assert 0 < CFG["split"]["val_fraction"] < 1
assert CFG["sam"]["prompt_batch"] >= 1 and CFG["sam"]["accumulate_images"] >= 1
assert min(CFG["search"]["det_conf"]) >= CFG["predict"]["proposal_conf"]
assert all(0 < x < 1 for x in CFG["search"]["mask_threshold"])
RESOURCES = CFG["resources"]
GPU_IDS = RESOURCES["gpu_ids"]
assert GPU_IDS and len(set(GPU_IDS)) == len(GPU_IDS)
assert RESOURCES["cpu_workers"] >= len(GPU_IDS)
assert RESOURCES["torch_threads"] >= 1 and RESOURCES["prefetch_factor"] >= 1
assert RESOURCES["vram_budget_gib"] > 0
assert RESOURCES["log_poll_seconds"] > 0 and CFG["sam"]["log_every_images"] >= 1
if os.name != "posix":
    raise RuntimeError("This resource profile uses Linux loader workers. Run on Kaggle with GPU T4 x2.")
if not torch.cuda.is_available() or max(GPU_IDS) >= torch.cuda.device_count():
    raise RuntimeError("Select GPU T4 x2 in Kaggle and restart the kernel, or explicitly use gpu_ids=[0].")
SIZE = CFG["native_size"]
AMP = CFG["amp"]
DEVICE = torch.device("cpu")           # orchestration/metrics stay on CPU in the notebook
PIPELINE_VERSION = "native-yolo-sam-kaggle-v2"
torch.set_num_threads(RESOURCES["torch_threads"])
cv2.setNumThreads(1)
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
settings.update({key: False for key in ("wandb", "mlflow", "clearml", "comet", "tensorboard")
                 if key in settings})
RUN = Path(CFG["output_root"]).expanduser().resolve() / f"{CFG['run_name']}_{MODE}"
RUN.mkdir(parents=True, exist_ok=True)

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def file_sha(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    os.replace(temporary, path)

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def amp_context():
    return torch.autocast("cuda", dtype=torch.float16) if AMP else nullcontext()

def assert_native(array):
    if tuple(array.shape[-2:]) != (SIZE, SIZE):
        raise ValueError(f"Expected native {SIZE}x{SIZE}; received {tuple(array.shape)}")

def read_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise OSError(f"Cannot read {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    assert_native(rgb.transpose(2, 0, 1))
    return rgb

def native_input_guard(module, args):
    assert_native(args[0])

versions = {x: importlib.metadata.version(x) for x in
            ("torch", "torchvision", "ultralytics", "numpy", "scipy", "pycocotools")}
devices = [{"id": gpu, "name": torch.cuda.get_device_name(gpu),
            "vram_gib": torch.cuda.get_device_properties(gpu).total_memory / 2**30} for gpu in GPU_IDS]
save_json(RUN / "environment.json", {"python": sys.version, "packages": versions, "devices": devices})
save_json(RUN / "resource_plan.json", {"pipeline_version": PIPELINE_VERSION,
    "cpu_workers_total": RESOURCES["cpu_workers"], "devices": devices,
    "training": {"yolo_gpu": GPU_IDS[0], "sam_gpu": GPU_IDS[-1]},
    "inference": "one complete YOLO + SAM replica per GPU; sources sharded without duplication",
    "vram_budget_gib_per_gpu": RESOURCES["vram_budget_gib"]})
print("Run directory:", RUN, "\nDevices:", devices,
      "\nTotal CPU worker budget:", RESOURCES["cpu_workers"])


## Kaggle resource scheduler

Preparation and CPU scoring use a bounded four-worker pool. Training runs the
independent detector on GPU 0 and SAM on GPU 1 concurrently, with two loader
workers each. SAM reuses one persistent pool for train and validation; YOLO uses
zero extra validation workers. Each prediction worker later holds its own complete
pipeline. Source images are split between GPUs without duplication, with two CPU
threads per GPU overlapping image prefetch and compressed-cache writes.

Fresh GPU interpreters receive definitions captured from this notebook as an
ignored JSON runtime bundle. No separate Python source files or notebook pickles
are needed. The loader workers use Linux `fork` for CPU-only reads; GPU processes
are always fresh interpreters. Logs are streamed into the running cell with `[train_yolo]` and `[train_sam]`
labels and are also retained under `logs/`. SAM reports progress within each epoch;
YOLO's verbose logs and progress rows appear as they are written. Log polling and
SAM's image interval are editable in the configuration cell. Periodic worker status
continues during quiet phases. Each T4 has its own 13 GiB allocator budget; memory is
never pooled. This two-T4 configuration has not been run on Kaggle here.

In [ ]:
# YOLO_SAM_RUNTIME
import codecs
import re
def ordered_map(function, items, workers=None):
    """Bound in-flight work and keep source order, including when jobs finish out of order."""
    workers = CFG["resources"]["cpu_workers"] if workers is None else workers
    if workers <= 0:
        yield from map(function, items)
        return
    iterator = iter(items)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = deque()
        for _ in range(workers):
            try: pending.append(pool.submit(function, next(iterator)))
            except StopIteration: break
        while pending:
            yield pending.popleft().result()
            try: pending.append(pool.submit(function, next(iterator)))
            except StopIteration: pass

def allocate_workers(total, jobs):
    if not 1 <= jobs <= total:
        raise ValueError("Need at least one CPU worker per active GPU job")
    return [total // jobs + (index < total % jobs) for index in range(jobs)]

def capture_runtime_sources(sources=None):
    """Export only imports/definitions from marked cells, never their run-all actions."""
    if sources is None:
        sources = get_ipython().history_manager.input_hist_raw[1:]
    statements = []
    for source in sources:
        if not source.startswith("# YOLO_SAM_RUNTIME"):
            continue
        statements.extend(node for node in ast.parse(source).body
                          if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.ClassDef)))
    if not statements:
        raise RuntimeError("Run the notebook cells in order; no runtime definitions were captured")
    return ast.unparse(ast.Module(body=statements, type_ignores=[]))

def gpu_job_environment(gpu, environ=None):
    env = dict(os.environ if environ is None else environ)
    visible = [x.strip() for x in env.get("CUDA_VISIBLE_DEVICES", "").split(",") if x.strip()]
    env["CUDA_VISIBLE_DEVICES"] = visible[gpu] if visible else str(gpu)
    for key in ("RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"):
        env.pop(key, None)
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[key] = str(CFG["resources"]["torch_threads"])
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    return env

def configure_worker(job):
    """A new interpreter sees exactly one physical GPU, always addressed as cuda:0."""
    globals().update(job["context"])
    globals().update(RUN=Path(job["context"]["run_dir"]), ROOT=Path(job["context"]["root_dir"]),
                     DEVICE=torch.device("cuda:0"), WORKER_CPU_JOBS=job["workers"])
    torch.set_num_threads(CFG["resources"]["torch_threads"])
    cv2.setNumThreads(1)
    # Ultralytics has separate image/label scanning thread pools; cap those too.
    import ultralytics.data.dataset as yolo_dataset_module
    import ultralytics.data.base as yolo_base_module
    yolo_dataset_module.NUM_THREADS = WORKER_CPU_JOBS
    yolo_base_module.NUM_THREADS = WORKER_CPU_JOBS
    # Plot synchronously in the GPU coordinator rather than creating extra threads.
    from functools import partial
    import ultralytics.models.yolo.detect.train as detect_train_module
    import ultralytics.models.yolo.detect.val as detect_val_module
    detect_train_module.plot_images = partial(detect_train_module.plot_images, threaded=False)
    detect_val_module.plot_images = partial(detect_val_module.plot_images, threaded=False)
    torch.cuda.set_device(0)
    total = torch.cuda.get_device_properties(0).total_memory
    fraction = min(0.90, CFG["resources"]["vram_budget_gib"] * 2**30 / total)
    torch.cuda.set_per_process_memory_fraction(fraction, 0)
    torch.backends.cudnn.benchmark = job["kind"] != "yolo" or not CFG["yolo"]["deterministic"]
    torch.backends.cuda.matmul.allow_tf32 = False  # T4 uses FP16 Tensor Cores
    random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
    from torchvision.ops import nms
    nms(torch.tensor([[0., 0., 1., 1.]], device=DEVICE), torch.ones(1, device=DEVICE), 0.5)
    globals().update(YOLO_DIR=RUN / "detector", YOLO_BEST=RUN / "detector/weights/best.pt",
                     SAM_BEST=RUN / "sam_best.pt", SAM_LAST=RUN / "sam_last.pt")
    print(f"Job {job['name']}: physical GPU {job['gpu']}, CPU workers {job['workers']}, "
          f"allocator cap {fraction * total / 2**30:.1f} GiB", flush=True)

def worker_main(job):
    configure_worker(job)
    if job["kind"] == "yolo":
        train_detector()
    elif job["kind"] == "sam":
        globals()["sam"] = NativeSAM2()
        probe_native_sam()
        train_sam()
    elif job["kind"] == "predict":
        predict_worker(job["paths"], job["category"])
    else:
        raise ValueError(f"Unknown GPU job {job['kind']}")

class JobLogTail:
    """Incremental log-file reader; no extra reader threads or blocking pipes."""
    def __init__(self, name, path):
        self.name = name
        self.reader = Path(path).open("rb")
        self.decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
        self.pending = ""
        self.finished = False

    def emit(self, line):
        # Strip terminal color/cursor escapes for readable notebook output.
        line = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", line).strip()
        if line:
            print(f"[{self.name}] {line}", flush=True)

    def pump(self, final=False):
        if self.finished:
            return
        data = self.reader.read() if final else self.reader.read(256 * 1024)
        self.pending += self.decoder.decode(data, final=final)
        pieces = re.split(r"(\r\n|\r|\n)", self.pending)
        progress = None
        for index in range(0, len(pieces) - 1, 2):
            line, ending = pieces[index:index + 2]
            if ending == "\r":
                if line.strip(): progress = line  # keep only latest overwritten progress row
            else:
                self.emit(line)
                progress = None
        self.pending = pieces[-1]
        if progress is not None:
            self.emit(progress)
        if final:
            self.emit(self.pending)              # preserve final output without a newline
            self.pending = ""
            self.finished = True

    def close(self):
        self.reader.close()


def launch_gpu_jobs(jobs):
    """Own and reap every subprocess; failures stop peers and preserve resumable artifacts."""
    if not jobs: return
    if len({job["gpu"] for job in jobs}) != len(jobs):
        raise ValueError("Concurrent jobs must use different GPUs")
    budget = allocate_workers(CFG["resources"]["cpu_workers"], len(jobs))
    source = capture_runtime_sources()
    context = {"CFG": CFG, "MODE": MODE, "SIZE": SIZE, "AMP": AMP,
        "run_dir": str(RUN), "root_dir": str(ROOT), "RUN_KEY": RUN_KEY,
        "train_records": train_records, "val_records": val_records,
        "RESOLVED_WEIGHTS": RESOLVED_WEIGHTS, "SAM_BASE_SHA": SAM_BASE_SHA, "SAM_KEY": SAM_KEY,
        "PIPELINE_VERSION": PIPELINE_VERSION}
    if "PROVENANCE" in globals():
        context.update(PROVENANCE=PROVENANCE, PREDICTION_KEY=PREDICTION_KEY)
    bootstrap = ("import json,sys; p=json.load(open(sys.argv[1],encoding='utf-8')); "
                 "exec(compile(p['source'],'<notebook-runtime>','exec'),globals()); worker_main(p['job'])")
    processes, handles, tails = [], [], []
    started = time.monotonic()
    next_update = started
    try:
        for job, workers in zip(jobs, budget):
            job = {**job, "workers": workers, "context": context}
            payload_path = RUN / "runtime" / f"{job['name']}.json"
            save_json(payload_path, {"source": source, "job": job})
            log_path = RUN / "logs" / f"{job['name']}.log"
            log_path.parent.mkdir(parents=True, exist_ok=True)
            handle = log_path.open("w", encoding="utf-8")
            handles.append(handle)
            process = subprocess.Popen([sys.executable, "-u", "-c", bootstrap, str(payload_path)],
                env=gpu_job_environment(job["gpu"]), stdout=handle, stderr=subprocess.STDOUT)
            processes.append((job["name"], process, log_path))
            if CFG["resources"].get("stream_logs", True):
                tails.append(JobLogTail(job["name"], log_path))
        while True:
            for tail in tails: tail.pump()
            states = [(name, process.poll()) for name, process, _ in processes]
            failed = next(((name, path) for name, process, path in processes
                           if process.returncode not in (None, 0)), None)
            if failed:
                tail = "\n".join(failed[1].read_text(encoding="utf-8", errors="replace").splitlines()[-25:])
                raise RuntimeError(f"GPU job {failed[0]} failed. See {failed[1]}\n{tail}")
            if all(code == 0 for _, code in states): break
            if time.monotonic() - started > CFG["resources"]["job_timeout_hours"] * 3600:
                raise TimeoutError("GPU stage timed out; inspect logs and resume its checkpoints")
            if time.monotonic() >= next_update:
                print({name: "running" if code is None else "done" for name, code in states}, flush=True)
                next_update = time.monotonic() + CFG["resources"]["progress_seconds"]
            time.sleep(CFG["resources"].get("log_poll_seconds", 0.5))
    finally:
        for _, process, _ in processes:
            if process.poll() is None: process.terminate()
        for _, process, _ in processes:
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
        try:
            for tail in tails: tail.pump(final=True)
        finally:
            for tail in tails: tail.close()
            for handle in handles: handle.close()

class SourceDataset(Dataset):
    """CPU-only native reads; no CUDA objects are created by loader workers."""
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        record = self.rows[index]
        rgb = read_rgb(record["path"])
        return {"index": index, "path": record["path"],
                "image": torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))),
                "target": load_json(record["target_path"]) if "target_path" in record else {}}

class ReplaySampler(Sampler):
    def __init__(self): self.indices = []
    def __iter__(self): return iter(self.indices)
    def __len__(self): return len(self.indices)

def one_sample(batch): return batch[0]

def seed_cpu_worker(index):
    torch.set_num_threads(1); cv2.setNumThreads(1)

def source_loader(rows, sampler):
    workers = WORKER_CPU_JOBS
    options = {"prefetch_factor": CFG["resources"]["prefetch_factor"],
               "multiprocessing_context": "fork"} if workers else {}
    return DataLoader(SourceDataset(rows), batch_size=1, sampler=sampler,
        num_workers=workers, persistent_workers=workers > 0, collate_fn=one_sample,
        pin_memory=CFG["resources"]["pin_memory"] and DEVICE.type == "cuda",
        worker_init_fn=seed_cpu_worker, **options)

## 2. Merge annotators, audit inputs, and split by source

Only `file_name` identifies annotator records of the same source. Pixel hashes also
keep identical images with different filenames in the same partition. By default,
multi-annotator sources stay in training, matching the existing U-Net policy.
All COCO categories become one foreground class. No dilation, hole filling,
polygon simplification, resizing, or cropping is applied to training masks.

The union is exact. Its 4/8-connected components define training instances; area
filtering defaults to one pixel. If components touch, they become one target.
Original annotations are retained separately for organizer-style validation.

In [ ]:
# YOLO_SAM_RUNTIME
ANNOTATION_FILE = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

def find_root():
    candidates = [Path(CFG["data_root"]).expanduser()] if CFG["data_root"] else []
    if not candidates:
        for ancestor in (Path.cwd(), *Path.cwd().parents):
            candidates.extend([ancestor, ancestor / "MAGFilo_1.0_Kaggle_2026"])
        if Path("/kaggle/input").exists():
            candidates.extend(p.parent.parent for p in Path("/kaggle/input").rglob(ANNOTATION_FILE))
    for candidate in candidates:
        if (candidate / "train" / ANNOTATION_FILE).is_file():
            if (candidate / "test/test_images").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set CFG['data_root'] to the directory containing train/ and test/.")

def encode_mask(mask):
    mask = np.asarray(mask)
    if mask.ndim != 2 or (mask.dtype != np.bool_ and not np.isin(mask, [0, 1]).all()):
        raise ValueError("Expected a binary HW mask")
    rle = coco_mask.encode(np.asfortranarray(mask, dtype=np.uint8))
    return {"size": list(mask.shape), "counts": rle["counts"].decode("ascii")}

def decode_mask(rle):
    return coco_mask.decode({"size": rle["size"],
                             "counts": rle["counts"].encode("ascii")}).astype(bool)

def annotation_mask(annotation, shape):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        if not segmentation:
            raise ValueError(f"Empty polygon: {annotation['id']}")
        for polygon in segmentation:
            xy = np.asarray(polygon, dtype=float)
            if xy.ndim != 1 or len(xy) < 6 or len(xy) % 2 or not np.isfinite(xy).all():
                raise ValueError(f"Invalid polygon: {annotation['id']}")
            xy = xy.reshape(-1, 2)
            if (xy < 0).any() or (xy[:, 0] > shape[1]).any() or (xy[:, 1] > shape[0]).any():
                raise ValueError(f"Polygon outside image: {annotation['id']}")
        rle = coco_mask.merge(coco_mask.frPyObjects(segmentation, *shape))
    elif isinstance(segmentation, dict):
        rle = dict(segmentation)
        if rle.get("size") != list(shape):
            raise ValueError("Annotation RLE dimensions differ from image")
        if isinstance(rle["counts"], list):
            counts = rle["counts"]
            if any(not isinstance(n, int) or n < 0 for n in counts) or sum(counts) != math.prod(shape):
                raise ValueError("Invalid uncompressed RLE")
            rle = coco_mask.frPyObjects(rle, *shape)
        elif isinstance(rle["counts"], str):
            rle["counts"] = rle["counts"].encode("ascii")
    else:
        raise ValueError("Unsupported annotation segmentation")
    mask = coco_mask.decode(rle).astype(bool)
    if mask.shape != shape or not mask.any():
        raise ValueError(f"Empty or wrong-sized annotation {annotation['id']}")
    return mask

def union_components(masks, shape, connectivity=8, min_area=1):
    union = np.zeros(shape, dtype=bool)
    for mask in masks:
        if mask.shape != shape:
            raise ValueError("Mask shape mismatch")
        union |= mask
    labels, count = ndi.label(union, ndi.generate_binary_structure(2, 2 if connectivity == 8 else 1))
    areas = np.bincount(labels.ravel(), minlength=count + 1)
    components = [encode_mask(labels == index) for index in range(1, count + 1)
                  if areas[index] >= min_area]
    return union, components

def rle_box(rle):
    x, y, width, height = coco_mask.toBbox(rle).tolist()
    return [x, y, x + width, y + height]

ROOT = find_root()
annotation_path = ROOT / "train" / ANNOTATION_FILE
payload = load_json(annotation_path)
images = payload["images"]
if len({str(x["id"]) for x in images}) != len(images):
    raise ValueError("Duplicate COCO image IDs")
if len({str(x["id"]) for x in payload["annotations"]}) != len(payload["annotations"]):
    raise ValueError("Duplicate annotation IDs")
known_ids = {str(x["id"]) for x in images}
annotations = defaultdict(list)
for annotation in payload["annotations"]:
    if str(annotation["image_id"]) not in known_ids or annotation.get("iscrowd", 0):
        raise ValueError("Unknown source or unsupported crowd annotation")
    annotations[str(annotation["image_id"])].append(annotation)
groups = defaultdict(list)
for image in images:
    if (image["height"], image["width"]) != (SIZE, SIZE):
        raise ValueError(f"Non-native COCO dimensions: {image['file_name']}")
    groups[image["file_name"]].append(image)

data_key = digest({"annotations": file_sha(annotation_path), "merge": CFG["merge"], "size": SIZE})
target_dir = RUN / "targets" / data_key[:16]
image_dir = (ROOT / "train/train_images").resolve()

def index_source(item):
    name, entries = item
    path = (image_dir / name).resolve()
    if not path.is_relative_to(image_dir) or Path(name).name != name:
        raise ValueError("Expected a plain source filename inside train_images")
    source_hash = hashlib.sha256(read_rgb(path).tobytes()).hexdigest()
    originals = [{"image_id": str(entry["id"]), "annotations": annotations[str(entry["id"])]}
                 for entry in entries]
    target_path = target_dir / f"{digest([name, source_hash])[:24]}.json"
    return {"name": name, "path": str(path), "source_hash": source_hash,
            "target_path": str(target_path), "originals": originals}

records = list(ordered_map(index_source, sorted(groups.items())))
print(f"Indexed and hashed {len(records)} native sources", flush=True)

by_hash = defaultdict(list)
for record in records:
    by_hash[record["source_hash"]].append(record)
eligible = sorted(key for key, rows in by_hash.items()
    if not CFG["split"]["multi_annotator_train_only"] or all(len(r["originals"]) == 1 for r in rows))
if len(eligible) < 2:
    raise ValueError("Need at least two independent eligible source groups")
np.random.default_rng(CFG["seed"]).shuffle(eligible)
n_val = min(len(eligible) - 1, max(1, round(len(records) * CFG["split"]["val_fraction"])))
validation_hashes = set(eligible[:n_val])
train_all = [r for r in records if r["source_hash"] not in validation_hashes]
val_all = [r for r in records if r["source_hash"] in validation_hashes]
assert not ({r["source_hash"] for r in train_all} & {r["source_hash"] for r in val_all})
train_records = train_all[:CFG["smoke"]["train_sources"]] if MODE == "smoke" else train_all
val_records = val_all[:CFG["smoke"]["val_sources"]] if MODE == "smoke" else val_all
assert train_records and val_records
active_records = train_records + val_records
def prepare_target(record):
    target_path = Path(record["target_path"])
    if not target_path.exists():
        masks = (annotation_mask(a, (SIZE, SIZE)) for entry in record["originals"] for a in entry["annotations"])
        union, instances = union_components(
            masks, (SIZE, SIZE), connectivity=CFG["merge"]["connectivity"],
            min_area=CFG["merge"]["min_component_area"])
        save_json(target_path, {"instances": instances, "union_pixels": int(union.sum()),
                  "boxes": [rle_box(rle) for rle in instances]})
    return len(load_json(target_path)["instances"])

component_counts = list(ordered_map(prepare_target, active_records))

manifest = {"data_key": data_key, "mode": MODE,
    "source_hashes": {r["name"]: r["source_hash"] for r in records},
    "train": [r["name"] for r in train_all], "val": [r["name"] for r in val_all],
    "active_train": [r["name"] for r in train_records], "active_val": [r["name"] for r in val_records]}
RUN_KEY = digest(manifest)
if (RUN / "split.json").exists() and load_json(RUN / "split.json") != manifest:
    raise RuntimeError("This run directory belongs to a different dataset/split. Change run_name.")
save_json(RUN / "split.json", manifest)
training_contract = {"pipeline": PIPELINE_VERSION, "run": RUN_KEY, "amp": AMP,
    "yolo": {k:v for k,v in CFG["yolo"].items() if k not in ("epochs", "resume", "workers")},
    "sam": {k:v for k,v in CFG["sam"].items() if k not in ("epochs", "resume", "log_every_images")}}
if (RUN / "training_contract.json").exists() and load_json(RUN / "training_contract.json") != training_contract:
    raise RuntimeError("Training settings changed. Use a new run_name to avoid stale checkpoints.")
save_json(RUN / "training_contract.json", training_contract)
save_json(RUN / "config.json", {"mode": MODE, **CFG})
save_json(RUN / "data_report.json", {
    "source_images": len(records), "annotator_records": len(images),
    "annotations": len(payload["annotations"]),
    "multi_annotator_sources": sum(len(r["originals"]) > 1 for r in records),
    "identical_pixel_groups": sum(len(rows) > 1 for rows in by_hash.values()),
    "active_merged_instances": sum(component_counts),
    "train_sources": len(train_records), "val_sources": len(val_records)})
print(load_json(RUN / "data_report.json"))

In [ ]:
# YOLO_SAM_RUNTIME
def preview_targets(record):
    rgb = read_rgb(record["path"])
    target = load_json(record["target_path"])
    union = np.zeros((SIZE, SIZE), bool)
    for rle in target["instances"]:
        union |= decode_mask(rle)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb); axes[0].set_title(record["name"])
    axes[1].imshow(rgb)
    colors = plt.get_cmap("tab10")
    for i, original in enumerate(record["originals"]):
        layer = np.zeros((SIZE, SIZE), bool)
        for annotation in original["annotations"]:
            layer |= annotation_mask(annotation, (SIZE, SIZE))
        axes[1].contour(layer, levels=[0.5], colors=[colors(i % 10)], linewidths=0.6)
    axes[1].set_title(f"{len(record['originals'])} annotator records")
    axes[2].imshow(rgb); axes[2].imshow(np.ma.masked_where(~union, union), alpha=0.65, cmap="autumn")
    axes[2].set_title(f"Union: {len(target['instances'])} training components")
    for axis in axes: axis.axis("off")
    fig.tight_layout(); fig.savefig(RUN / "merge_preview.png", dpi=150, bbox_inches="tight"); plt.show()

preview_targets(next((r for r in train_records if len(r["originals"]) > 1), train_records[0]))

## 3. Native YOLO detection training

Images are copied byte for byte; labels contain boxes of union components, not
approximated polygons. Spatial augmentations that scale, crop, mosaic, translate,
or warp are disabled. Flips preserve every pixel and image dimensions.
`imgsz=2048`, fixed image batch size and a model input hook enforce native input.
The requested CUDA AMP setting remains enabled; the trainer bypasses only
Ultralytics' unrelated 640-pixel compatibility probe. Training and validation forwards are guarded at 2048 pixels.

YOLO `best.pt` is selected by detector validation metrics. SAM's best decoder below
is selected by held-out box-prompt loss; the final combined system is then scored
with actual detector prompts. Smoke mode uses one epoch for each model.

This cell defines the detector worker. Training starts after both model workers
are defined, so the independent YOLO and SAM jobs can overlap on separate T4s.

YOLO uses the requested AdamW optimizer, cosine decay, deterministic training,
0.5 horizontal/vertical flips, brightness jitter `hsv_v=0.3`, pretrained weights,
plots and verbose logs. Hue/saturation jitter and all geometric warps stay off.
Cache mode, batch size, epochs, patience and detection cap remain editable in CFG.


In [ ]:
# YOLO_SAM_RUNTIME
from copy import copy
from ultralytics.data.build import build_yolo_dataset, build_dataloader
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator

class NativeDetectionValidator(DetectionValidator):
    def preprocess(self, batch):
        batch = super().preprocess(batch)
        assert_native(batch["img"])
        return batch

class NativeDetectionTrainer(DetectionTrainer):
    def _setup_train(self):
        # Keep the requested amp=True in the training arguments, but skip the
        # SDK's unrelated 640px compatibility probe. Real training stays native.
        # This temporary change is confined to this fresh, single-GPU interpreter.
        import ultralytics.engine.trainer as trainer_module
        previous = trainer_module.check_amp
        trainer_module.check_amp = lambda model: bool(AMP and self.device.type == "cuda")
        try:
            return super()._setup_train()
        finally:
            trainer_module.check_amp = previous

    def build_dataset(self, img_path, mode="train", batch=None):
        # Default rectangular validation can add 32px padding to square images.
        # Fixed square loading ensures both train and validation are exactly 2048.
        stride = max(int(self.model.stride.max()), 32)
        return build_yolo_dataset(self.args, img_path, batch, self.data,
                                  mode=mode, rect=False, stride=stride)

    def get_dataloader(self, dataset_path, batch_size=16, rank=-1, mode="train"):
        dataset = self.build_dataset(dataset_path, mode, batch_size)
        return build_dataloader(dataset, batch=batch_size,
            workers=WORKER_CPU_JOBS if mode == "train" else 0,
            shuffle=mode == "train", rank=-1, device=self.device,
            pin_memory=CFG["resources"]["pin_memory"])

    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)
        assert_native(batch["img"])
        return batch

    def get_validator(self):
        return NativeDetectionValidator(self.test_loader, save_dir=self.save_dir,
                                        args=copy(self.args), _callbacks=self.callbacks)

def export_yolo_dataset():
    destination = RUN / "detector_data"
    for partition, rows in (("train", train_records), ("val", val_records)):
        image_out = destination / "images" / partition
        label_out = destination / "labels" / partition
        image_out.mkdir(parents=True, exist_ok=True)
        label_out.mkdir(parents=True, exist_ok=True)
        expected = {r["name"] for r in rows}
        if any(p.name not in expected for p in image_out.iterdir()):
            raise RuntimeError("Stale detector data. Use a fresh run_name.")
        def export_record(record):
            copied = image_out / record["name"]
            if not copied.exists() or file_sha(copied) != file_sha(record["path"]):
                shutil.copy2(record["path"], copied)
            boxes = load_json(record["target_path"])["boxes"]
            lines = [f"0 {(x1+x2)/(2*SIZE):.10f} {(y1+y2)/(2*SIZE):.10f} "
                     f"{(x2-x1)/SIZE:.10f} {(y2-y1)/SIZE:.10f}" for x1, y1, x2, y2 in boxes]
            (label_out / f"{copied.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        for _ in ordered_map(export_record, rows): pass
    # JSON is valid YAML and avoids introducing a separate YAML writer.
    yaml_path = destination / "data.yaml"
    save_json(yaml_path, {"path": destination.as_posix(), "train": "images/train",
                         "val": "images/val", "names": {"0": "filament"}})
    return yaml_path

def guard_detector_train(trainer):
    model = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
    model.model[0].register_forward_pre_hook(native_input_guard)

def train_detector():
    data_yaml = RUN / "detector_data/data.yaml"
    if YOLO_BEST.exists() and not CFG["yolo"]["resume"]:
        print("Reusing existing detector best.pt; change run_name to retrain.")
    else:
        resume_path = YOLO_DIR / "weights/last.pt"
        if CFG["yolo"]["resume"] and not resume_path.exists():
            raise FileNotFoundError("No detector last.pt in this run to resume")
        detector = YOLO(str(resume_path) if CFG["yolo"]["resume"] else RESOLVED_WEIGHTS["yolo"])
        detector.add_callback("on_train_start", guard_detector_train)
        train_results = detector.train(
            trainer=NativeDetectionTrainer,
            data=str(data_yaml), imgsz=SIZE, epochs=1 if MODE == "smoke" else CFG["yolo"]["epochs"],
            batch=CFG["yolo"]["batch"], workers=WORKER_CPU_JOBS, device=str(DEVICE),
            optimizer=CFG["yolo"]["optimizer"], lr0=CFG["yolo"]["lr0"],
            weight_decay=CFG["yolo"]["weight_decay"], patience=CFG["yolo"]["patience"],
            project=str(RUN), name="detector", exist_ok=True, resume=CFG["yolo"]["resume"],
            seed=CFG["seed"], deterministic=CFG["yolo"]["deterministic"], amp=(DEVICE.type == "cuda" and AMP),
            pretrained=CFG["yolo"]["pretrained"], cos_lr=CFG["yolo"]["cos_lr"],
            max_det=CFG["predict"]["max_det"], verbose=CFG["yolo"]["verbose"],
            channels_last=CFG["yolo"]["channels_last"], compile=False,
            multi_scale=False, rect=False, cache=CFG["yolo"]["cache"], val=True, plots=CFG["yolo"]["plots"],
            mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0, close_mosaic=0,
            degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
            hsv_h=0.0, hsv_s=0.0, hsv_v=CFG["yolo"]["hsv_v"],
            fliplr=CFG["yolo"]["flip_lr"], flipud=CFG["yolo"]["flip_ud"],
        )
        metrics = {key: float(value) for key, value in train_results.results_dict.items()}
        save_json(RUN / "yolo_training_metrics.json", metrics)
        del detector
        gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 4. SAM 2 with native input and trainable mask decoder

`set_imgsz((2048,2048))` updates prompt coordinates and embedding grid dimensions.
We normalize RGB pixels directly and run the encoder on that tensor. Features have
spatial sizes 512, 256 and 128; the mask decoder produces 512-pixel logits which
are upsampled for loss against the **unchanged native masks**. Box prompts always
use original pixel coordinates. Boxes guide decoding; they never crop the image.

The image encoder, prompt encoder, video memory modules, and decoder's encoder-side
`conv_s0/conv_s1` projections are frozen. The remaining decoder parameters train.
Each image is encoded once, then every component is supervised in small prompt
batches. Image accumulation controls memory without dropping components.

SAM training uses one persistent two-worker loader for both training and
validation. The following launch cell starts both model jobs concurrently.

In [ ]:
# YOLO_SAM_RUNTIME
class NativeSAM2:
    """Small adapter around the pinned Ultralytics SAM 2 implementation."""
    def __init__(self):
        self.model = SAM(RESOLVED_WEIGHTS["sam"]).model.to(DEVICE).float()
        if not hasattr(self.model, "sam_prompt_encoder") or self.model.backbone_stride != 16:
            raise ValueError("Choose a SAM 2 / SAM 2.1 checkpoint")
        self.model.set_imgsz((SIZE, SIZE))
        self.model.requires_grad_(False)
        self.model.sam_mask_decoder.requires_grad_(True)
        for name in ("conv_s0", "conv_s1"):
            getattr(self.model.sam_mask_decoder, name).requires_grad_(False)
        self.model.eval()
        self.model.image_encoder.register_forward_pre_hook(native_input_guard)
        self.mean = torch.tensor([123.675, 116.28, 103.53], device=DEVICE).view(1, 3, 1, 1)
        self.std = torch.tensor([58.395, 57.12, 57.375], device=DEVICE).view(1, 3, 1, 1)

    def trainable(self):
        return [p for p in self.model.parameters() if p.requires_grad]

    def encode(self, rgb):
        if isinstance(rgb, np.ndarray):
            image = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1)))[None]
        else:
            image = rgb[None] if rgb.ndim == 3 else rgb
        image = image.to(DEVICE, dtype=torch.float32, non_blocking=True)
        assert_native(image)
        image = (image - self.mean) / self.std
        # no_grad (rather than inference_mode) lets decoder backward save these tensors.
        with torch.no_grad(), amp_context():
            output = self.model.forward_image(image)
            _, features, _, sizes = self.model._prepare_backbone_features(output)
            if self.model.directly_add_no_mem_embed:
                features[-1] = features[-1] + self.model.no_mem_embed
            maps = [f.permute(1, 2, 0).reshape(1, -1, *size) for f, size in zip(features, sizes)]
        assert [tuple(m.shape[-2:]) for m in maps] == [(SIZE//4, SIZE//4), (SIZE//8, SIZE//8), (SIZE//16, SIZE//16)]
        return {"embedding": maps[-1], "high_res": maps[:-1]}

    def decode(self, features, boxes):
        boxes = torch.as_tensor(boxes, device=DEVICE, dtype=torch.float32).reshape(-1, 2, 2)
        labels = torch.tensor([[2, 3]], device=DEVICE, dtype=torch.int32).expand(len(boxes), -1)
        with torch.no_grad():
            sparse, dense = self.model.sam_prompt_encoder(points=(boxes, labels), boxes=None, masks=None)
            positional = self.model.sam_prompt_encoder.get_dense_pe()
        masks, quality, _, _ = self.model.sam_mask_decoder(
            image_embeddings=features["embedding"], image_pe=positional,
            sparse_prompt_embeddings=sparse, dense_prompt_embeddings=dense,
            multimask_output=False, repeat_image=len(boxes) > 1,
            high_res_features=features["high_res"],
        )
        return masks[:, 0], quality[:, 0]

def padded_boxes(boxes, jitter=0, rng=None):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
    boxes += np.array([-1, -1, 1, 1], np.float32) * CFG["sam"]["box_pad"]
    if jitter:
        boxes += rng.uniform(-jitter, jitter, boxes.shape).astype(np.float32)
    boxes[:, :2] = np.clip(boxes[:, :2], 0, SIZE - 1)
    boxes[:, 2:] = np.clip(boxes[:, 2:], 0, SIZE)
    boxes[:, 2:] = np.maximum(boxes[:, 2:], boxes[:, :2] + 1)
    return boxes

def native_loss(low_logits, predicted_quality, truth):
    logits = F.interpolate(low_logits[:, None].float(), (SIZE, SIZE), mode="bilinear", align_corners=False)[:, 0]
    assert_native(truth)
    probability = logits.sigmoid()
    positive_weight = torch.tensor(CFG["sam"]["positive_weight"], device=DEVICE)
    bce = F.binary_cross_entropy_with_logits(logits, truth, pos_weight=positive_weight)
    intersection = (probability * truth).sum((1, 2))
    dice = (1 - (2 * intersection + 1) / (probability.sum((1, 2)) + truth.sum((1, 2)) + 1)).mean()
    with torch.no_grad():
        hard = logits > 0
        actual_iou = ((hard * truth).sum((1, 2)) / ((hard | truth.bool()).sum((1, 2)).clamp_min(1)))
    quality_loss = F.mse_loss(predicted_quality.float(), actual_iou)
    return (CFG["sam"]["bce_weight"] * bce + CFG["sam"]["dice_weight"] * dice
            + CFG["sam"]["iou_weight"] * quality_loss)

def save_checkpoint(path, epoch, optimizer, scaler, best, bad_epochs):
    temporary = path.with_suffix(".tmp")
    torch.save({"signature": SAM_KEY, "mode": MODE, "epoch": epoch,
        "decoder": sam.model.sam_mask_decoder.state_dict(), "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(), "best_loss": best, "bad_epochs": bad_epochs}, temporary)
    os.replace(temporary, path)

def read_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    if checkpoint["signature"] != SAM_KEY or checkpoint["mode"] != MODE:
        raise RuntimeError("SAM checkpoint does not match data, settings, base weights or run mode")
    return checkpoint

def probe_native_sam():
    # Always probe a real native image with one ground-truth box, including when YOLO
    # has no detections during a very short smoke run. This is never a test prediction.
    probe_record = next(r for r in train_records if load_json(r["target_path"])["instances"])
    probe_target = load_json(probe_record["target_path"])
    with torch.no_grad(), amp_context():
        probe_features = sam.encode(read_rgb(probe_record["path"]))
        probe_logits, _ = sam.decode(probe_features, padded_boxes(probe_target["boxes"][:1]))
    assert tuple(probe_logits.shape[-2:]) == (SIZE // 4, SIZE // 4)
    save_json(RUN / "native_probe.json", {"source": probe_record["name"],
              "encoder_input": [1, 3, SIZE, SIZE], "decoder_logits": list(probe_logits.shape),
              "note": "Real pretrained SAM 2 forward with a training target box, not a test prediction."})
    del probe_features, probe_logits
    print("Native SAM 2 probe passed; trainable parameters:", sum(p.numel() for p in sam.trainable()))

In [ ]:
# YOLO_SAM_RUNTIME
def image_loss_steps(record, training, rng, sample=None):
    """Yield prompt losses; only prompt_batch native masks reside on the GPU."""
    target = sample["target"] if sample is not None else load_json(record["target_path"])
    n_objects = len(target["instances"])
    if not n_objects:
        return
    rgb = sample["image"] if sample is not None else read_rgb(record["path"])
    flip_lr = training and rng.random() < CFG["sam"]["flip_lr"]
    flip_ud = training and rng.random() < CFG["sam"]["flip_ud"]
    if torch.is_tensor(rgb):
        if flip_lr: rgb = rgb.flip(2)
        if flip_ud: rgb = rgb.flip(1)
    else:
        if flip_lr: rgb = rgb[:, ::-1]
        if flip_ud: rgb = rgb[::-1]
    features = sam.encode(rgb)
    boxes = np.asarray(target["boxes"], dtype=np.float32)
    if flip_lr: boxes[:, [0, 2]] = SIZE - boxes[:, [2, 0]]
    if flip_ud: boxes[:, [1, 3]] = SIZE - boxes[:, [3, 1]]
    boxes = padded_boxes(boxes, CFG["sam"]["box_jitter"] if training else 0, rng)
    order = rng.permutation(n_objects) if training else np.arange(n_objects)
    for start in range(0, n_objects, CFG["sam"]["prompt_batch"]):
        indices = order[start:start + CFG["sam"]["prompt_batch"]]
        masks = np.stack([decode_mask(target["instances"][int(i)]) for i in indices])
        if flip_lr: masks = masks[:, :, ::-1]
        if flip_ud: masks = masks[:, ::-1]
        truth = torch.from_numpy(np.ascontiguousarray(masks)).to(DEVICE).float()
        with amp_context():
            logits, quality = sam.decode(features, boxes[indices])
        loss = native_loss(logits, quality, truth)
        yield loss, len(indices) / n_objects

def validate_sam(loader, sampler):
    sam.model.eval()
    sampler.indices = list(range(len(train_records), len(train_records) + len(val_records)))
    total = 0.0
    with torch.no_grad():
        for val_index, sample in enumerate(loader, 1):
            record = val_records[sample["index"] - len(train_records)]
            for loss, fraction in image_loss_steps(record, False, np.random.default_rng(CFG["seed"]), sample):
                total += float(loss) * fraction
            interval = CFG["sam"].get("log_every_images", 16)
            if val_index % interval == 0 or val_index == len(val_records):
                print(f"SAM validation {val_index}/{len(val_records)} images: "
                      f"loss={total / val_index:.5f}", flush=True)
    return total / len(val_records)

def train_sam():
    if SAM_BEST.exists() and not CFG["sam"]["resume"]:
        read_checkpoint(SAM_BEST)
        print("Reusing existing SAM checkpoint", flush=True)
        return
    sampler = ReplaySampler()
    loader = source_loader(train_records + val_records, sampler)
    optimizer = torch.optim.AdamW(sam.trainable(), lr=CFG["sam"]["lr"], weight_decay=CFG["sam"]["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    start, best, bad_epochs = 0, float("inf"), 0
    history_path = RUN / "sam_history.csv"
    history = []
    if CFG["sam"]["resume"]:
        checkpoint = read_checkpoint(SAM_LAST)
        sam.model.sam_mask_decoder.load_state_dict(checkpoint["decoder"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        start, best, bad_epochs = checkpoint["epoch"] + 1, checkpoint["best_loss"], checkpoint["bad_epochs"]
        if history_path.exists():
            history = [row for row in pd.read_csv(history_path).to_dict("records") if row["epoch"] < start]
    epochs = 1 if MODE == "smoke" else CFG["sam"]["epochs"]
    accumulation = CFG["sam"]["accumulate_images"]
    for epoch in range(start, epochs):
        started = time.perf_counter()
        rng = np.random.default_rng(CFG["seed"] + epoch)
        order = rng.permutation(len(train_records))
        sampler.indices = order.tolist()
        iterator = iter(loader)
        sam.model.eval()
        sam.model.sam_mask_decoder.train()
        train_total = 0.0
        print(f"SAM epoch {epoch+1}/{epochs}: training {len(order)} native images", flush=True)
        for group_start in range(0, len(order), accumulation):
            group = order[group_start:group_start + accumulation]
            optimizer.zero_grad(set_to_none=True)
            had_loss = False
            for index in group:
                sample = next(iterator)
                if sample["index"] != int(index): raise RuntimeError("Loader/sampler order mismatch")
                for loss, fraction in image_loss_steps(train_records[int(index)], True, rng, sample):
                    if not torch.isfinite(loss):
                        raise FloatingPointError("Non-finite SAM loss")
                    train_total += float(loss.detach()) * fraction
                    scaler.scale(loss * fraction / len(group)).backward()
                    had_loss = True
            if had_loss:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(sam.trainable(), CFG["sam"]["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
            processed = group_start + len(group)
            interval = CFG["sam"].get("log_every_images", 16)
            if processed == len(order) or processed // interval > group_start // interval:
                print(f"SAM epoch {epoch+1}/{epochs}: {processed}/{len(order)} images, "
                      f"loss={train_total / processed:.5f}, lr={optimizer.param_groups[0]['lr']:.2g}, "
                      f"elapsed={time.perf_counter() - started:.1f}s", flush=True)
        print(f"SAM epoch {epoch+1}/{epochs}: validating", flush=True)
        val_loss = validate_sam(loader, sampler)
        if not math.isfinite(val_loss):
            raise FloatingPointError("Non-finite validation loss")
        improved = val_loss < best
        best = min(best, val_loss)
        bad_epochs = 0 if improved else bad_epochs + 1
        row = {"epoch": epoch, "train_loss": train_total / len(train_records),
               "val_loss": val_loss, "seconds": time.perf_counter() - started}
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        save_checkpoint(SAM_LAST, epoch, optimizer, scaler, best, bad_epochs)
        if improved: save_checkpoint(SAM_BEST, epoch, optimizer, scaler, best, bad_epochs)
        print(row, flush=True)
        if bad_epochs >= CFG["sam"]["patience"]:
            break
    del optimizer, scaler


In [ ]:
# YOLO_SAM_RUNTIME
from ultralytics.utils.downloads import attempt_download_asset

RESOLVED_WEIGHTS = {name: str(Path(attempt_download_asset(CFG[name]["weights"])).resolve())
                    for name in ("yolo", "sam")}
SAM_BASE_SHA = file_sha(RESOLVED_WEIGHTS["sam"])
SAM_KEY = digest({"pipeline": PIPELINE_VERSION, "run": RUN_KEY, "base": SAM_BASE_SHA, "size": SIZE,
    "sam_settings": {k:v for k,v in CFG["sam"].items() if k not in ("resume", "epochs", "weights", "log_every_images")}})
YOLO_DIR, YOLO_BEST = RUN / "detector", RUN / "detector/weights/best.pt"
SAM_BEST, SAM_LAST = RUN / "sam_best.pt", RUN / "sam_last.pt"
save_json(RUN / "resolved_weights.json", RESOLVED_WEIGHTS)
jobs = []
if CFG["stages"]["train_yolo"] and (not YOLO_BEST.exists() or CFG["yolo"]["resume"]):
    export_yolo_dataset()
    jobs.append({"name": "train_yolo", "kind": "yolo", "gpu": GPU_IDS[0]})
if CFG["stages"]["train_sam"] and (not SAM_BEST.exists() or CFG["sam"]["resume"]):
    jobs.append({"name": "train_sam", "kind": "sam", "gpu": GPU_IDS[-1]})
if len(GPU_IDS) == 1:
    for job in jobs: launch_gpu_jobs([job])
else:
    launch_gpu_jobs(jobs)
if not YOLO_BEST.is_file() or not SAM_BEST.is_file():
    raise FileNotFoundError("Train both models first; this run's best checkpoint is missing")
checkpoint = torch.load(SAM_BEST, map_location="cpu", weights_only=True)
if checkpoint["signature"] != SAM_KEY or checkpoint["mode"] != MODE:
    raise RuntimeError("SAM checkpoint does not match this experiment")
del checkpoint
if (RUN / "sam_history.csv").exists():
    pd.read_csv(RUN / "sam_history.csv").plot(x="epoch", y=["train_loss", "val_loss"], title="Native SAM decoder loss")
    plt.tight_layout(); plt.savefig(RUN / "sam_loss.png", dpi=150); plt.show()

## Training metrics and grouped artifact gallery

Both GPU training jobs have finished before this section runs. The tables show
YOLO's returned validation metrics, last-epoch losses, best recorded box mAP, and
SAM decoder losses. The gallery uses the groups/layout in the configuration cell.
Only available plots are loaded, so a short smoke run can contain fewer batches.
YOLO26 is anchor-free; the batch group displays the generated training images.
SAM mask quality is evaluated in the combined validation section below.

In [ ]:
# YOLO_SAM_RUNTIME
from IPython.display import display, Markdown

def display_training_metrics(run_dir):
    run_dir = Path(run_dir)
    metrics_path = RUN / "yolo_training_metrics.json"
    if metrics_path.is_file():
        display(Markdown("### YOLO: validation metrics returned after training"))
        display(pd.DataFrame(load_json(metrics_path).items(), columns=["metric", "value"]))
    history_path = run_dir / "results.csv"
    if history_path.is_file():
        history = pd.read_csv(history_path)
        history.columns = history.columns.str.strip()
        display(Markdown("### YOLO: last epoch losses and metrics"))
        display(history.tail(1).T.rename(columns={history.index[-1]: "last_epoch"}))
        quality_column = "metrics/mAP50-95(B)"
        if quality_column in history.columns and history[quality_column].notna().any():
            display(Markdown("### YOLO: highest recorded box mAP50?95"))
            display(history.loc[[history[quality_column].idxmax()]])
    if (RUN / "sam_history.csv").is_file():
        display(Markdown("### SAM: decoder training and validation loss"))
        history = pd.read_csv(RUN / "sam_history.csv")
        display(history.tail())
        if history["val_loss"].notna().any():
            display(Markdown("Best recorded SAM prompt-validation loss:"))
            display(history.loc[[history["val_loss"].idxmin()]])


def display_yolo_gallery(run_dir):
    run_dir = Path(run_dir)
    print(f"Loading generated artifact visuals from: {run_dir}")
    for group, files in YOLO_VISUAL_GROUPS.items():
        available = [name for name in files if (run_dir / name).is_file()]
        if not available:
            if group == "Instance Segmentation Mask Metrics":
                print("SAM mask metrics (PQ, Dice, IoU) appear after combined validation. "
                      "YOLO Mask* curves are generated by YOLO segmentation models.")
            continue
        display(Markdown(f"### {group}"))
        layout = DISPLAY_SETTINGS[group]
        cols = layout["cols"]
        rows = math.ceil(len(available) / cols)
        fig, axes = plt.subplots(rows, cols, squeeze=False,
            figsize=(layout["col_width"] * cols, layout["row_height"] * rows))
        for axis, name in zip(axes.flat, available):
            bgr = cv2.imread(str(run_dir / name))
            if bgr is None: raise OSError(f"Cannot decode training artifact: {run_dir / name}")
            axis.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
            axis.set_title(name, fontsize=11, fontweight="bold", pad=8)
            axis.axis("off")
        for axis in list(axes.flat)[len(available):]: axis.axis("off")
        fig.tight_layout(); plt.show(); plt.close(fig)


display_training_metrics(YOLO_DIR)
display_yolo_gallery(YOLO_DIR)

## 5. Combined prediction, caching and organizer-compatible evaluation

YOLO supplies padded box prompts. SAM encodes each full image once, decodes boxes
in prompt batches, and stores compact low-resolution logits on CPU/disk. Filtering
upsamples logits to native size, thresholds, and performs mask IoU NMS. It preserves
disconnected pieces of one predicted instance and permits overlapping instances.
It never truncates a predicted mask to the prompt box. No generic minimum-size
cleanup, dilation or largest-component filtering removes narrow structures.

Validation compares the same source prediction to **each original annotator record**.
It counts every pair with IoU strictly greater than 0.5, plus unmatched prediction
columns and GT rows, then sums globally, matching the supplied organizer notebook.
It also reports merged-foreground Dice/IoU as diagnostics, separately from PQ.

In [ ]:
# YOLO_SAM_RUNTIME
PROVENANCE = {"pipeline": PIPELINE_VERSION, "run": RUN_KEY, "mode": MODE, "native_size": SIZE,
    "yolo_sha256": file_sha(YOLO_BEST), "sam_sha256": file_sha(SAM_BEST),
    "sam_base_sha256": SAM_BASE_SHA, "predict": CFG["predict"], "box_pad": CFG["sam"]["box_pad"]}
PREDICTION_KEY = digest(PROVENANCE)

@torch.no_grad()
def predict_raw(path, image=None):
    rgb = read_rgb(path) if image is None else image
    # Tensor source bypasses all YOLO image resizing/letterbox preprocessing.
    if torch.is_tensor(rgb):
        pixels = rgb.to(DEVICE, dtype=torch.float32, non_blocking=True)[None]
    else:
        pixels = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).to(DEVICE).float()[None]
    tensor = pixels / 255.0
    result = detector.predict(source=tensor, imgsz=SIZE, device=str(DEVICE), half=AMP,
        conf=CFG["predict"]["proposal_conf"], iou=CFG["predict"]["box_iou"],
        max_det=CFG["predict"]["max_det"], augment=False, verbose=False)[0]
    boxes = result.boxes.xyxy.float().cpu().numpy()
    scores = result.boxes.conf.float().cpu().numpy()
    logits = []
    if len(boxes):
        features = sam.encode(pixels)
        prompts = padded_boxes(boxes)
        for start in range(0, len(boxes), CFG["sam"]["prompt_batch"]):
            with amp_context():
                predicted, _ = sam.decode(features, prompts[start:start + CFG["sam"]["prompt_batch"]])
            logits.extend(predicted.float().cpu().numpy().astype(np.float16))
    low_size = SIZE // 4
    return {"logits": np.stack(logits) if logits else np.empty((0, low_size, low_size), np.float16),
            "scores": scores, "boxes": boxes}

def raw_cache_path(path, category):
    key = digest({"prediction": PREDICTION_KEY, "image": file_sha(path)})
    return RUN / "prediction_cache" / category / f"{key}.npz"

def cached_raw(path, category):
    cache = raw_cache_path(path, category)
    if not cache.is_file():
        raise FileNotFoundError(f"Run the GPU cache stage before CPU scoring: {cache}")
    with np.load(cache, allow_pickle=False) as content:
        return {k: content[k] for k in content.files}

def read_prediction_image(path):
    rgb = read_rgb(path)
    image = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1)))
    if CFG["resources"]["pin_memory"]:
        image = image.pin_memory()
    return path, image


def write_raw_cache(path, raw, category):
    cache = raw_cache_path(path, category)
    cache.parent.mkdir(parents=True, exist_ok=True)
    with cache.with_suffix(".tmp").open("wb") as handle:
        np.savez_compressed(handle, **raw)
    os.replace(cache.with_suffix(".tmp"), cache)


def predict_worker(paths, category):
    globals()["detector"] = YOLO(str(YOLO_BEST))
    detector.model.model[0].register_forward_pre_hook(native_input_guard)
    globals()["sam"] = NativeSAM2()
    sam.model.sam_mask_decoder.load_state_dict(read_checkpoint(SAM_BEST)["decoder"])
    sam.model.eval()
    # The same bounded thread pool overlaps one native-image read and cache
    # compression/writes with GPU inference; it creates no extra loader processes.
    with ThreadPoolExecutor(max_workers=WORKER_CPU_JOBS) as pool:
        iterator = iter(paths)
        next_image = pool.submit(read_prediction_image, next(iterator)) if paths else None
        writes = deque()
        for index in range(len(paths)):
            path, image = next_image.result()
            upcoming = next(iterator, None)
            next_image = pool.submit(read_prediction_image, upcoming) if upcoming is not None else None
            raw = predict_raw(path, image)
            writes.append(pool.submit(write_raw_cache, path, raw, category))
            if len(writes) >= WORKER_CPU_JOBS:
                writes.popleft().result()
            print(f"{category}: {index+1}/{len(paths)} sources decoded, {len(raw['scores'])} proposals", flush=True)
        for future in writes: future.result()

def cache_prediction_sources(paths, category):
    paths = [str(path) for path in paths]
    # Byte-identical sources share one cache entry, so schedule that entry once.
    unique = {}
    for path in paths:
        cache = raw_cache_path(path, category)
        if not cache.is_file(): unique.setdefault(str(cache), path)
    pending = list(unique.values())
    shards = [pending[index::len(GPU_IDS)] for index in range(len(GPU_IDS))]
    jobs = [{"name": f"{category}_gpu_{gpu}", "kind": "predict", "gpu": gpu,
             "paths": shard, "category": category} for gpu, shard in zip(GPU_IDS, shards) if shard]
    launch_gpu_jobs(jobs)
    if any(not raw_cache_path(path, category).is_file() for path in paths):
        raise RuntimeError("Incomplete GPU prediction cache")

def filter_raw(raw, parameters):
    instances = []
    for index in np.argsort(-raw["scores"], kind="stable"):
        if raw["scores"][index] < parameters["det_conf"]:
            continue
        # Only mask logits are interpolated; source images and target masks never are.
        low = torch.from_numpy(raw["logits"][index].astype(np.float32))[None, None]
        logit_cutoff = math.log(parameters["mask_threshold"] / (1 - parameters["mask_threshold"]))
        mask = F.interpolate(low, (SIZE, SIZE), mode="bilinear", align_corners=False)[0, 0].numpy() > logit_cutoff
        if int(mask.sum()) < parameters["min_area"]:
            continue
        rle = encode_mask(mask)
        if instances:
            overlaps = coco_mask.iou([rle], instances, [0] * len(instances))[0]
            if np.any(overlaps > parameters["mask_nms_iou"]):
                continue
        instances.append(rle)
    return instances

def pq_counts(gt, predicted):
    iou = (np.asarray(coco_mask.iou(gt, predicted, [0] * len(predicted)), dtype=float)
           if gt and predicted else np.zeros((len(gt), len(predicted))))
    hits = iou > 0.5
    return {"tp": int(hits.sum()), "fp": int((~hits.any(axis=0)).sum()),
            "fn": int((~hits.any(axis=1)).sum()), "iou_sum": float(iou[hits].sum())}

def aggregate_pq(counts):
    total = {key: sum(row[key] for row in counts) for key in ("tp", "fp", "fn", "iou_sum")}
    denominator = total["tp"] + 0.5 * (total["fp"] + total["fn"])
    return {**total, "pq": total["iou_sum"] / denominator if denominator else 0.0}

def score_source(record, predicted):
    rows = []
    for original in record["originals"]:
        gt = [encode_mask(annotation_mask(a, (SIZE, SIZE))) for a in original["annotations"]]
        rows.append({"source": record["name"], "image_id": original["image_id"], **pq_counts(gt, predicted)})
    return rows

def foreground_metrics(record, predicted):
    gt = np.zeros((SIZE, SIZE), bool)
    pred = np.zeros_like(gt)
    for rle in load_json(record["target_path"])["instances"]: gt |= decode_mask(rle)
    for rle in predicted: pred |= decode_mask(rle)
    intersection = int((gt & pred).sum())
    union = int((gt | pred).sum())
    return {"source": record["name"], "intersection": intersection,
            "union": union, "sum_areas": int(gt.sum() + pred.sum())}

cache_prediction_sources([record["path"] for record in val_records], "val")
keys = list(CFG["search"])
trial_parameters = [dict(zip(keys, values)) for values in
                    itertools.product(*(CFG["search"][key] for key in keys))]

def score_trials_for_source(record):
    raw = cached_raw(record["path"], "val")  # only one source per CPU worker in memory
    return [aggregate_pq(score_source(record, filter_raw(raw, parameters))) for parameters in trial_parameters]

trial_counts = [[] for _ in trial_parameters]
for results in ordered_map(score_trials_for_source, val_records):
    for bucket, result in zip(trial_counts, results): bucket.append(result)
trials = [{**parameters, **aggregate_pq(counts)}
          for parameters, counts in zip(trial_parameters, trial_counts)]
trials_df = pd.DataFrame(trials).sort_values("pq", ascending=False, kind="stable")
trials_df.to_csv(RUN / "threshold_trials.csv", index=False)
winner = trials_df.iloc[0]
SELECTED = {key: float(winner[key]) for key in keys}
SELECTED["min_area"] = int(SELECTED["min_area"])
save_json(RUN / "selected_postprocessing.json", {"provenance": PROVENANCE,
    "prediction_key": PREDICTION_KEY, "parameters": SELECTED, "validation_pq": float(winner["pq"])})

def selected_source_metrics(record):
    prediction = filter_raw(cached_raw(record["path"], "val"), SELECTED)
    return score_source(record, prediction), foreground_metrics(record, prediction)

validation_rows, foreground_rows = [], []
for rows, foreground in ordered_map(selected_source_metrics, val_records):
    validation_rows.extend(rows); foreground_rows.append(foreground)
summary = aggregate_pq(validation_rows)
intersection = sum(row["intersection"] for row in foreground_rows)
union = sum(row["union"] for row in foreground_rows)
areas = sum(row["sum_areas"] for row in foreground_rows)
summary.update(foreground_iou=intersection / union if union else 0.0,
               foreground_dice=2 * intersection / areas if areas else 0.0,
               sources=len(val_records), annotator_records=len(validation_rows), mode=MODE)
save_json(RUN / "validation_summary.json", summary)
pd.DataFrame(validation_rows).to_csv(RUN / "validation_per_annotator.csv", index=False)
pd.DataFrame(foreground_rows).to_csv(RUN / "validation_foreground.csv", index=False)
display(trials_df.head())
print("Selected parameters:", SELECTED, "\nValidation:", summary)

display(Markdown("### Combined YOLO + SAM validation metrics"))
display(pd.DataFrame([summary]))


In [ ]:
# YOLO_SAM_RUNTIME
def show_prediction(record, predicted, destination):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    rgb = read_rgb(record["path"])
    for axis in axes: axis.imshow(rgb); axis.axis("off")
    for rle in load_json(record["target_path"])["instances"]:
        axes[0].contour(decode_mask(rle), levels=[0.5], colors=["cyan"], linewidths=0.6)
    for index, rle in enumerate(predicted):
        axes[1].contour(decode_mask(rle), levels=[0.5], colors=[plt.get_cmap("tab20")(index % 20)], linewidths=0.6)
    axes[0].set_title("Merged training-target contours")
    axes[1].set_title(f"YOLO + SAM: {len(predicted)} predicted instances")
    fig.suptitle(record["name"]); fig.tight_layout()
    fig.savefig(destination, dpi=150, bbox_inches="tight"); plt.show()

review_dir = RUN / "review"
review_dir.mkdir(exist_ok=True)
for record in val_records[:CFG["review_images"]]:
    prediction = filter_raw(cached_raw(record["path"], "val"), SELECTED)
    show_prediction(record, prediction, review_dir / f"{Path(record['name']).stem}.png")

## 6. Submission with native compressed COCO RLE

The competition CSV has exactly `filament_id,segmentation_rle`. Each nonempty mask
becomes `<original_test_stem>_<1-based instance number>`. RLE uses Fortran ordering
and compressed COCO counts, not a custom run-length format. Every emitted mask is
decoded and checked for an exact round trip at 2048 × 2048.

An image with no predictions has **no CSV rows**, and an explicit zero count in the
manifest. Every test source must have a completed journal before the CSV is replaced
atomically. Journals include image and model fingerprints so reruns can resume safely.
Smoke mode only emits `smoke_submission.csv`; only full mode can emit `submission.csv`.

In [ ]:
# YOLO_SAM_RUNTIME
COLUMNS = ["filament_id", "segmentation_rle"]

def submission_rows(stem, instances):
    rows = []
    for index, rle in enumerate(instances, 1):
        mask = decode_mask(rle)
        assert_native(mask)
        if not mask.any():
            raise ValueError("Empty predicted instance")
        canonical = encode_mask(mask)
        if canonical != rle:
            raise ValueError("Submission RLE round trip failed")
        rows.append({"filament_id": f"{stem}_{index}", "segmentation_rle": rle["counts"]})
    return rows

def create_submission():
    selection = load_json(RUN / "selected_postprocessing.json")
    if selection["prediction_key"] != PREDICTION_KEY or selection["provenance"] != PROVENANCE:
        raise RuntimeError("Selected thresholds belong to different model checkpoints")
    test_paths = sorted(p for p in (ROOT / "test/test_images").iterdir()
                        if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if not test_paths or len({p.stem for p in test_paths}) != len(test_paths):
        raise ValueError("Missing test images or duplicate source stems")
    if MODE == "smoke": test_paths = test_paths[:CFG["smoke"]["test_sources"]]
    submission_key = digest({"provenance": PROVENANCE, "parameters": selection["parameters"]})
    journals = RUN / "submission_journal" / submission_key[:16]
    journals.mkdir(parents=True, exist_ok=True)
    # Completed journals need no raw logits; interrupted test inference is sharded.
    def journal_path(path):
        return journals / f"{digest({'name': path.name, 'sha256': file_sha(path), 'submission': submission_key})}.json"
    uncached = [path for path in test_paths if not journal_path(path).is_file()]
    cache_prediction_sources(uncached, "test")

    def encode_source(path):
        # Validate even a cached source's dimensions before accepting its journal.
        read_rgb(path)
        image_key = digest({"name": path.name, "sha256": file_sha(path), "submission": submission_key})
        journal = journals / f"{image_key}.json"
        if not journal.exists():
            raw = cached_raw(path, "test")
            rows = submission_rows(path.stem, filter_raw(raw, selection["parameters"]))
            save_json(journal, {"key": image_key, "rows": rows})
        entry = load_json(journal)
        if entry["key"] != image_key:
            raise ValueError("Corrupt submission journal")
        for instance_index, row in enumerate(entry["rows"], 1):
            if set(row) != set(COLUMNS) or row["filament_id"] != f"{path.stem}_{instance_index}":
                raise ValueError("Invalid journal columns or source/instance ID")
            mask = decode_mask({"size": [SIZE, SIZE], "counts": row["segmentation_rle"]})
            if not mask.any() or encode_mask(mask)["counts"] != row["segmentation_rle"]:
                raise ValueError("Invalid journal RLE")
        return path.name, entry["rows"]

    all_rows, counts = [], {}
    for index, (name, rows) in enumerate(ordered_map(encode_source, test_paths)):
        counts[name] = len(rows)
        all_rows.extend(rows)
        print(f"Test {index+1}/{len(test_paths)}: {name}, {counts[name]} instances", flush=True)
    ids = [row["filament_id"] for row in all_rows]
    if len(set(ids)) != len(ids) or len(counts) != len(test_paths):
        raise ValueError("Incomplete source coverage or duplicate filament IDs")
    output = RUN / ("submission.csv" if MODE == "full" else "smoke_submission.csv")
    temporary = output.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=COLUMNS)
        writer.writeheader(); writer.writerows(all_rows)
    os.replace(temporary, output)
    report = {"mode": MODE, "submission_key": submission_key, "provenance": PROVENANCE,
        "parameters": selection["parameters"], "csv_sha256": file_sha(output),
        "sources": len(test_paths), "instances": len(all_rows),
        "zero_prediction_sources": [name for name, count in counts.items() if count == 0],
        "instances_per_source": counts}
    save_json(output.with_suffix(".manifest.json"), report)
    print("Saved:", output, "\nInstances:", len(all_rows), "Zero-prediction sources:", len(report["zero_prediction_sources"]))
    return output

if CFG["stages"]["submission"]:
    submission_path = create_submission()

## 7. Fast contract checks

These checks exercise the delicate data/metric contracts without a second training
run: union across annotators, corner connectivity, exact-threshold PQ, overlapping
GT, empty cases, native input enforcement, and compressed RLE orientation. Smoke
training, the native SAM probe and submission round trips supply execution evidence.

In [ ]:
# YOLO_SAM_RUNTIME
def run_contract_checks():
    a = np.zeros((12, 16), bool); a[1:5, 2:8] = True
    b = np.zeros_like(a); b[3:7, 6:11] = True
    c = np.zeros_like(a); c[9:11, 13:15] = True
    union, parts = union_components([a, b, c], a.shape)
    assert np.array_equal(union, a | b | c) and len(parts) == 2
    recovered = np.zeros_like(a)
    for rle in parts: recovered |= decode_mask(rle)
    assert np.array_equal(recovered, union)
    diagonal = np.eye(2, dtype=bool)
    assert len(union_components([diagonal], diagonal.shape, 4)[1]) == 2
    assert len(union_components([diagonal], diagonal.shape, 8)[1]) == 1
    assert np.array_equal(decode_mask(encode_mask(a)), a)
    polygon = {"id": "polygon", "segmentation": [[2, 1, 8, 1, 8, 5, 2, 5]]}
    assert np.array_equal(annotation_mask(polygon, a.shape), a)
    assert np.array_equal(annotation_mask({"id": "compressed", "segmentation": encode_mask(a)}, a.shape), a)
    runs, previous, length = [], 0, 0
    for pixel in a.ravel(order="F"):
        if int(pixel) == previous:
            length += 1
        else:
            runs.append(length); previous = int(pixel); length = 1
    runs.append(length)
    uncompressed = {"id": "uncompressed", "segmentation": {"size": list(a.shape), "counts": runs}}
    assert np.array_equal(annotation_mask(uncompressed, a.shape), a)
    one = np.zeros((2, 4), bool); one[0, 0] = True
    two = one.copy(); two[0, 1] = True
    assert pq_counts([encode_mask(one)], [encode_mask(two)])["tp"] == 0  # exactly 0.5
    overlap = pq_counts([encode_mask(a), encode_mask(a)], [encode_mask(a)])
    assert overlap["tp"] == 2 and overlap["fp"] == overlap["fn"] == 0
    assert aggregate_pq([overlap])["pq"] == 1.0
    assert pq_counts([], [encode_mask(a)])["fp"] == 1
    assert pq_counts([encode_mask(a)], [])["fn"] == 1
    assert aggregate_pq([pq_counts([], [])])["pq"] == 0.0
    try:
        assert_native(np.zeros((1024, 1024), np.uint8))
    except ValueError:
        pass
    else:
        raise AssertionError("Native input guard accepted a resized image")
    native = np.zeros((SIZE, SIZE), bool); native[7:9, 100:1700] = True
    rows = submission_rows("source_with_underscores", [encode_mask(native)])
    assert rows[0]["filament_id"] == "source_with_underscores_1"
    assert submission_rows("empty", []) == []
    print("All data, native-resolution, PQ and submission contracts passed.")

run_contract_checks()

## Reading the artifacts and continuing experiments

`config.json`, `environment.json`, `split.json` and `data_report.json` describe the
run. `merge_preview.png` audits target policy. `detector/weights/{best,last}.pt`
and `sam_{best,last}.pt` hold checkpoints. `sam_history.csv` and detector
`results.csv` contain training histories. `native_probe.json` records a real native
SAM forward. `threshold_trials.csv`, `validation_summary.json`, per-annotator results,
and `review/` support model review. `selected_postprocessing.json` binds thresholds to
checkpoint hashes. Cached logits and per-test journals make interrupted inference
resumable. The CSV manifest includes every processed test image, including zeros.

Set stage flags false to reuse this run's checkpoints. For an interrupted training
run, set the corresponding `resume=True`; ordinary reruns reuse completed best
checkpoints. YOLO's final completed runs cannot be extended with `resume`; use a new
experiment. Change `run_name` whenever changing data, targets or training settings.
Smoke and full artifacts live in separate directories. Smoke scores do not estimate
competition performance, and a header-only smoke CSV can occur after one epoch.

Sources verified for the pinned API: [Ultralytics training settings](https://docs.ultralytics.com/modes/train/),
[SAM 2 model size configuration](https://github.com/ultralytics/ultralytics/blob/v8.4.198/ultralytics/models/sam/modules/sam.py),
[SAM 2 predictor implementation](https://github.com/ultralytics/ultralytics/blob/v8.4.198/ultralytics/models/sam/predict.py),
and [Meta SAM 2](https://github.com/facebookresearch/sam2). The repository's supplied
`self-evaluation-notebook.ipynb` is the scoring reference. See the adjacent README
and `PIPELINE.md` for setup, tuning, memory tradeoffs and limitations.

The Kaggle profile also records `resource_plan.json`, `resolved_weights.json`,
per-job `logs/*.log` and notebook-derived `runtime/*.json` bundles. The notebook
kernel orchestrates CPU work and does not retain GPU models. Independent YOLO/SAM
training jobs run concurrently, not as one synchronized two-GPU model. If one
training job finishes first, its GPU waits for the other; prediction later uses
both complete replicas. Hardware throughput and the 13 GiB allocator budget have
not been tested on dual T4s here. Review `VALIDATION.md` for the separate CPU checks.
